In [1]:
import time
import math
import re
import random
from collections import defaultdict, Counter
import numpy as np
from datasets import load_dataset

def split_into_sentences(text):
    delimiters = ['.', '।', '॥', '?', '!']
    for d in delimiters:
        text = text.replace(d, f"{d}<SPLIT>")
    sentences = [s.strip() for s in text.split("<SPLIT>") if len(s.strip()) > 10]
    return sentences

def prepare_indiccorp_dataset(languages, samples_per_lang=1500, train_ratio=0.8, val_ratio=0.1):
    dataset_dict = defaultdict(list)
    for lang in languages:
        print(f"Loading data for language: {lang}...")
        try:
            data_url = f"https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/{lang}.txt"
            ds = load_dataset("text", data_files=data_url, split="train", streaming=True)
            sentences = []
            
            for item in ds:
                text = item.get("text", "")
                if not text:
                    continue
                extracted = split_into_sentences(text)
                sentences.extend(extracted)
                if len(sentences) >= samples_per_lang:
                    break
            
            selected = sentences[:samples_per_lang]
            random.seed(42)
            random.shuffle(selected)
            dataset_dict[lang] = selected
            print(f" -> Successfully collected {len(selected)} sentences.")
        except Exception as e:
            print(f" -> Failed to load language {lang}: {e}")

    train_data, val_data, test_data = [], [], []
    train_labels, val_labels, test_labels = [], [], []
    n_train = int(samples_per_lang * train_ratio)
    n_val = int(samples_per_lang * val_ratio)
    
    for lang, sents in dataset_dict.items():
        train_sents = sents[:n_train]
        val_sents = sents[n_train:n_train + n_val]
        test_sents = sents[n_train + n_val:]
        
        train_data.extend(train_sents)
        train_labels.extend([lang] * len(train_sents))
        val_data.extend(val_sents)
        val_labels.extend([lang] * len(val_sents))
        test_data.extend(test_sents)
        test_labels.extend([lang] * len(test_sents))
        
    return (train_data, train_labels), (val_data, val_labels), (test_data, test_labels)

In [2]:
INDIC_LANGUAGES = [
    "as", "bd", "bn", "dg", "gom", "gu", "hi-1", "kha", "kn", "ks", 
    "mai", "ml", "mni", "mr", "ne", "or", "pa", "sa", "sat", "sd", 
    "ta", "te", "ur"
]

(train_x, train_y), (val_x, val_y), (test_x, test_y) = prepare_indiccorp_dataset(
    languages=INDIC_LANGUAGES,
    samples_per_lang=1500,
    train_ratio=0.8,
    val_ratio=0.1
)

if len(train_x) == 0:
    raise ValueError("Dataset is empty! Check internet connection.")

Loading data for language: as...
 -> Successfully collected 1500 sentences.
Loading data for language: bd...
 -> Successfully collected 1500 sentences.
Loading data for language: bn...
 -> Successfully collected 1500 sentences.
Loading data for language: dg...
 -> Successfully collected 1500 sentences.
Loading data for language: gom...
 -> Successfully collected 1500 sentences.
Loading data for language: gu...
 -> Successfully collected 1500 sentences.
Loading data for language: hi-1...
 -> Successfully collected 1500 sentences.
Loading data for language: kha...
 -> Successfully collected 1500 sentences.
Loading data for language: kn...
 -> Successfully collected 1500 sentences.
Loading data for language: ks...
 -> Successfully collected 1500 sentences.
Loading data for language: mai...
 -> Successfully collected 1500 sentences.
Loading data for language: ml...
 -> Successfully collected 1500 sentences.
Loading data for language: mni...
 -> Successfully collected 1500 sentences.
Loadin

In [3]:
class CustomTfidfVectorizer:
    def __init__(self, max_features=15000, tf_type='raw', idf_type='unnormalized'):
        self.max_features = max_features
        self.tf_type = tf_type
        self.idf_type = idf_type
        self.vocabulary_ = {}
        self.idf_ = {}
        
    def _extract_ngrams(self, text):
        tokens = re.findall(r'\w+', text.lower())
        ngrams = []
        ngrams.extend(tokens)
        for i in range(len(tokens) - 1):
            ngrams.append(f"{tokens[i]}_{tokens[i+1]}")
        char_text = f"#{text.lower()}#"
        for n in (2, 3, 4):
            for i in range(len(char_text) - n + 1):
                ngrams.append(char_text[i:i+n])
        return ngrams

    def fit(self, raw_documents):
        df_counts = Counter()
        doc_count = len(raw_documents)
        
        for doc in raw_documents:
            unique_terms = set(self._extract_ngrams(doc))
            for term in unique_terms:
                df_counts[term] += 1
                
        most_common = df_counts.most_common(self.max_features)
        self.vocabulary_ = {term: idx for idx, (term, _) in enumerate(most_common)}
        
        max_df = most_common[0][1] if most_common else 0
        
        for term, idx in self.vocabulary_.items():
            df = df_counts[term]
            if self.idf_type == 'unnormalized':
                self.idf_[idx] = math.log((1 + doc_count) / (1 + df)) + 1.0
            elif self.idf_type == 'normalized':
                self.idf_[idx] = math.log((1 + max_df) / (1 + df)) + 1.0
                
        return self

    def transform(self, raw_documents):
        num_docs = len(raw_documents)
        vocab_size = len(self.vocabulary_)
        X = np.zeros((num_docs, vocab_size), dtype=np.float32)
        
        for doc_idx, doc in enumerate(raw_documents):
            ngrams = self._extract_ngrams(doc)
            term_counts = Counter(ngrams)
            total_terms = len(ngrams)
            
            if total_terms == 0:
                continue
                
            max_term_count = max(term_counts.values()) if term_counts else 1
                
            for term, count in term_counts.items():
                if term in self.vocabulary_:
                    feature_idx = self.vocabulary_[term]
                    
                    if self.tf_type == 'raw':
                        tf = count
                    elif self.tf_type == 'total_words':
                        tf = count / total_terms
                    elif self.tf_type == 'max_freq':
                        tf = count / max_term_count
                        
                    X[doc_idx, feature_idx] = tf * self.idf_[feature_idx]
        return X

    def fit_transform(self, raw_documents):
        self.fit(raw_documents)
        return self.transform(raw_documents)

class CustomLogisticRegression:
    def __init__(self, learning_rate=0.1, epochs=50, batch_size=256, l2_reg=1e-4):
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2_reg = l2_reg
        self.W = None
        self.b = None

    def _softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        num_classes = len(self.classes_)
        num_samples, num_features = X.shape
        
        label_to_idx = {label: i for i, label in enumerate(self.classes_)}
        y_encoded = np.array([label_to_idx[label] for label in y])
        Y_onehot = np.zeros((len(y), num_classes), dtype=np.float32)
        Y_onehot[np.arange(len(y)), y_encoded] = 1.0

        np.random.seed(42)
        self.W = np.random.randn(num_features, num_classes) * 0.01
        self.b = np.zeros((1, num_classes))

        for epoch in range(self.epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            
            for start_idx in range(0, num_samples, self.batch_size):
                batch_idx = indices[start_idx:start_idx + self.batch_size]
                X_batch = X[batch_idx]
                Y_batch = Y_onehot[batch_idx]
                m = len(batch_idx)

                Z = np.dot(X_batch, self.W) + self.b
                P = self._softmax(Z)

                dW = (1 / m) * np.dot(X_batch.T, (P - Y_batch)) + self.l2_reg * self.W
                db = (1 / m) * np.sum(P - Y_batch, axis=0, keepdims=True)

                np.clip(dW, -1.0, 1.0, out=dW)
                np.clip(db, -1.0, 1.0, out=db)

                self.W -= self.lr * dW
                self.b -= self.lr * db

    def predict(self, X):
        Z = np.dot(X, self.W) + self.b
        P = self._softmax(Z)
        return self.classes_[np.argmax(P, axis=1)]

def evaluate_macro_f1_silent(y_true, y_pred):
    classes = sorted(list(set(y_true)))
    f1_scores = []
    for c in classes:
        tp = sum((yt == c and yp == c) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != c and yp == c) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == c and yp != c) for yt, yp in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0
        f1_scores.append(f1)
    return sum(f1_scores) / len(f1_scores)

In [4]:
class CustomTfidfVectorizer:
    def __init__(self, max_features=15000, tf_type='raw', idf_type='unnormalized'):
        self.max_features = max_features
        self.tf_type = tf_type
        self.idf_type = idf_type
        self.vocabulary_ = {}
        self.idf_ = {}
        
    def _extract_ngrams(self, text):
        tokens = re.findall(r'\w+', text.lower())
        ngrams = []
        ngrams.extend(tokens)
        for i in range(len(tokens) - 1):
            ngrams.append(f"{tokens[i]}_{tokens[i+1]}")
        char_text = f"#{text.lower()}#"
        for n in (2, 3, 4):
            for i in range(len(char_text) - n + 1):
                ngrams.append(char_text[i:i+n])
        return ngrams

    def fit(self, raw_documents):
        df_counts = Counter()
        doc_count = len(raw_documents)
        
        for doc in raw_documents:
            unique_terms = set(self._extract_ngrams(doc))
            for term in unique_terms:
                df_counts[term] += 1
                
        most_common = df_counts.most_common(self.max_features)
        self.vocabulary_ = {term: idx for idx, (term, _) in enumerate(most_common)}
        
        max_df = most_common[0][1] if most_common else 0
        
        for term, idx in self.vocabulary_.items():
            df = df_counts[term]
            if self.idf_type == 'unnormalized':
                self.idf_[idx] = math.log((1 + doc_count) / (1 + df)) + 1.0
            elif self.idf_type == 'normalized':
                self.idf_[idx] = math.log((1 + max_df) / (1 + df)) + 1.0
                
        return self

    def transform(self, raw_documents):
        num_docs = len(raw_documents)
        vocab_size = len(self.vocabulary_)
        X = np.zeros((num_docs, vocab_size), dtype=np.float32)
        
        for doc_idx, doc in enumerate(raw_documents):
            ngrams = self._extract_ngrams(doc)
            term_counts = Counter(ngrams)
            total_terms = len(ngrams)
            
            if total_terms == 0:
                continue
                
            max_term_count = max(term_counts.values()) if term_counts else 1
                
            for term, count in term_counts.items():
                if term in self.vocabulary_:
                    feature_idx = self.vocabulary_[term]
                    
                    if self.tf_type == 'raw':
                        tf = count
                    elif self.tf_type == 'total_words':
                        tf = count / total_terms
                    elif self.tf_type == 'max_freq':
                        tf = count / max_term_count
                        
                    X[doc_idx, feature_idx] = tf * self.idf_[feature_idx]
        return X

    def fit_transform(self, raw_documents):
        self.fit(raw_documents)
        return self.transform(raw_documents)

class CustomLogisticRegression:
    def __init__(self, learning_rate=0.1, epochs=50, batch_size=256, l2_reg=1e-4):
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2_reg = l2_reg
        self.W = None
        self.b = None

    def _softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        num_classes = len(self.classes_)
        num_samples, num_features = X.shape
        
        label_to_idx = {label: i for i, label in enumerate(self.classes_)}
        y_encoded = np.array([label_to_idx[label] for label in y])
        Y_onehot = np.zeros((len(y), num_classes), dtype=np.float32)
        Y_onehot[np.arange(len(y)), y_encoded] = 1.0

        np.random.seed(42)
        self.W = np.random.randn(num_features, num_classes) * 0.01
        self.b = np.zeros((1, num_classes))

        for epoch in range(self.epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            
            for start_idx in range(0, num_samples, self.batch_size):
                batch_idx = indices[start_idx:start_idx + self.batch_size]
                X_batch = X[batch_idx]
                Y_batch = Y_onehot[batch_idx]
                m = len(batch_idx)

                Z = np.dot(X_batch, self.W) + self.b
                P = self._softmax(Z)

                dW = (1 / m) * np.dot(X_batch.T, (P - Y_batch)) + self.l2_reg * self.W
                db = (1 / m) * np.sum(P - Y_batch, axis=0, keepdims=True)

                np.clip(dW, -1.0, 1.0, out=dW)
                np.clip(db, -1.0, 1.0, out=db)

                self.W -= self.lr * dW
                self.b -= self.lr * db

    def predict(self, X):
        Z = np.dot(X, self.W) + self.b
        P = self._softmax(Z)
        return self.classes_[np.argmax(P, axis=1)]

def evaluate_macro_f1_silent(y_true, y_pred):
    classes = sorted(list(set(y_true)))
    f1_scores = []
    for c in classes:
        tp = sum((yt == c and yp == c) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != c and yp == c) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == c and yp != c) for yt, yp in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0
        f1_scores.append(f1)
    return sum(f1_scores) / len(f1_scores)

In [5]:
tf_variants = ['raw', 'total_words', 'max_freq']
idf_variants = ['unnormalized', 'normalized']

results = []

print(f"\n{'TF Normalization':<18} | {'IDF Normalization':<18} | {'Macro-F1 Score':<15} | {'Compute Time'}")
print("-" * 75)

for tf_val in tf_variants:
    for idf_val in idf_variants:
        start_time = time.time()
        
        # Using 10,000 max_features so the 6 loops complete in a reasonable time
        vectorizer = CustomTfidfVectorizer(max_features=10000, tf_type=tf_val, idf_type=idf_val)
        
        X_train = vectorizer.fit_transform(train_x)
        X_test = vectorizer.transform(test_x)
        
        model = CustomLogisticRegression(learning_rate=0.5, epochs=40, batch_size=256)
        model.fit(X_train, train_y)
        
        test_preds = model.predict(X_test)
        f1 = evaluate_macro_f1_silent(test_y, test_preds)
        
        elapsed = time.time() - start_time
        results.append((tf_val, idf_val, f1))
        
        print(f"{tf_val:<18} | {idf_val:<18} | {f1:<15.4f} | {elapsed:.1f}s")

print("-" * 75)


TF Normalization   | IDF Normalization  | Macro-F1 Score  | Compute Time
---------------------------------------------------------------------------
raw                | unnormalized       | 0.9517          | 95.7s
raw                | normalized         | 0.9528          | 95.1s
total_words        | unnormalized       | 0.9169          | 94.9s
total_words        | normalized         | 0.9221          | 95.0s
max_freq           | unnormalized       | 0.9525          | 95.6s
max_freq           | normalized         | 0.9544          | 95.7s
---------------------------------------------------------------------------
